In [0]:
%python
from pyspark.sql.functions import count, avg, min, max

# --------------------------------
# GOLD LAYER
# --------------------------------

# Silver Delta path
silver_path = "/Volumes/etlpipelineincome/default/bronze/silver/clean_income"

# Gold output paths
gold_customer_path = "/Volumes/etlpipelineincome/default/bronze/gold/customer_segmentation"

gold_summary_path = "/Volumes/etlpipelineincome/default/bronze/gold/cluster_summary"


# --------------------------------
# 1. READ SILVER
# --------------------------------

df_gold = spark.read \
    .format("delta") \
    .load(silver_path)


# --------------------------------
# 2. CREATE BUSINESS-READY TABLE
# --------------------------------

gold_customer = df_gold.select(
    "CustomerID",
    "Income",
    "SpendingScore",
    "KMeans_Cluster"
)

gold_customer.show(20)


# --------------------------------
# 3. CLUSTER SUMMARY
# --------------------------------

cluster_summary = gold_customer.groupBy(
    "KMeans_Cluster"
).agg(
    count("*").alias("Customer_Count"),
    avg("Income").alias("Average_Income"),
    avg("SpendingScore").alias("Average_SpendingScore"),
    min("Income").alias("Minimum_Income"),
    max("Income").alias("Maximum_Income")
).orderBy(
    "KMeans_Cluster"
)


# --------------------------------
# 4. SHOW GOLD SUMMARY
# --------------------------------

cluster_summary.show()


# --------------------------------
# 5. SAVE CUSTOMER GOLD TABLE
# --------------------------------

gold_customer.write \
    .format("delta") \
    .mode("overwrite") \
    .save(gold_customer_path)


# --------------------------------
# 6. SAVE CLUSTER SUMMARY
# --------------------------------

cluster_summary.write \
    .format("delta") \
    .mode("overwrite") \
    .save(gold_summary_path)


print("Gold layer created successfully")

+----------+------+-------------+--------------+
|CustomerID|Income|SpendingScore|KMeans_Cluster|
+----------+------+-------------+--------------+
|        19| 36023|           20|             2|
|        29| 25311|           14|             2|
|        53| 76886|           41|             1|
|        66| 28792|           12|             2|
|        68|128557|           29|             0|
|        78| 43483|           37|             2|
|        84|126530|           48|             0|
|       110| 31534|           18|             2|
|       149| 33986|           27|             2|
|       158| 72251|           43|             1|
|       161| 50535|           45|             1|
|       200| 70015|           16|             1|
|        18| 80263|           66|             0|
|        28| 87969|           18|             1|
|        35| 48693|           20|             2|
|        49|104654|           41|             0|
|        69| 93969|           57|             0|
|        74|141729| 

In [0]:
%python
summary_csv_path = "/Volumes/etlpipelineincome/default/bronze/gold/cluster_summary_csv"

cluster_summary.write \
    .mode("overwrite") \
    .option("header", True) \
    .csv(summary_csv_path)

print("Cluster summary saved as CSV successfully")

Cluster summary saved as CSV successfully
